# Furniture image-to-3D benchmark

This notebook is an isolated R&D harness. It does **not** import, modify, or replace the production TripoSR/FastAPI/Supabase workflow. It is preconfigured for **Lightning AI Studio** with SPAR3D low-VRAM mode, while retaining Colab compatibility. Run one model per Studio/runtime and keep the same input manifest for comparisons.

Automatic fallback is deliberately conservative: **TRELLIS.2 (>=24 GiB currently free VRAM) -> Hunyuan3D 2.1 full PBR (>=29 GiB) -> Stable Fast 3D (>=6 GiB)**. It never falls back to TripoSR. SPAR3D is available only as an explicit optional experiment.

Official sources: [TRELLIS.2 repository](https://github.com/microsoft/TRELLIS.2), [TRELLIS.2 model card](https://huggingface.co/microsoft/TRELLIS.2-4B), [Hunyuan3D 2.1 repository](https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1), [Hunyuan3D 2.1 paper](https://arxiv.org/abs/2506.15442), [SF3D repository](https://github.com/Stability-AI/stable-fast-3d), [SF3D model card](https://huggingface.co/stabilityai/stable-fast-3d), [SPAR3D repository](https://github.com/Stability-AI/stable-point-aware-3d), and [SPAR3D model card](https://huggingface.co/stabilityai/stable-point-aware-3d).

> No benchmark scores in this notebook are pre-filled. Values produced by executed cells are measured; hardware requirements and published timings shown in explanatory text are expected/published values.

## 1. Environment and GPU inspection

Run this before any installation. Selection uses **currently free** VRAM, not the marketing capacity of the GPU. A nominal 24 GB L4 commonly has less than 24 GiB usable/free and therefore must not install TRELLIS.2.

In [ ]:
import csv
import hashlib
import json
import os
import platform
import re
import shutil
import subprocess
import sys
from pathlib import Path

LIGHTNING_STUDIO_ROOT = Path('/teamspace/studios/this_studio')
IS_LIGHTNING = bool(os.environ.get('LIGHTNING_CLOUD_SPACE_ID')) or LIGHTNING_STUDIO_ROOT.is_dir()
IS_COLAB = bool(os.environ.get('COLAB_RELEASE_TAG')) or (Path('/content').is_dir() and not IS_LIGHTNING)
PLATFORM_NAME = 'lightning' if IS_LIGHTNING else 'colab' if IS_COLAB else 'linux'

def capture(command):
    try:
        completed = subprocess.run(command, capture_output=True, text=True, check=False)
    except FileNotFoundError as exc:
        return 127, str(exc)
    return completed.returncode, (completed.stdout + completed.stderr).strip()

def bytes_to_gib(value):
    return round(value / (1024 ** 3), 2)

def read_meminfo():
    values = {}
    path = Path('/proc/meminfo')
    if path.is_file():
        for line in path.read_text().splitlines():
            key, _, rest = line.partition(':')
            if rest:
                values[key] = int(rest.strip().split()[0]) * 1024
    return values

gpu_query = [
    'nvidia-smi',
    '--query-gpu=index,name,memory.total,memory.free,driver_version,compute_cap',
    '--format=csv,noheader,nounits',
]
gpu_rc, gpu_text = capture(gpu_query)
if gpu_rc != 0:
    gpu_query = [
        'nvidia-smi',
        '--query-gpu=index,name,memory.total,memory.free,driver_version',
        '--format=csv,noheader,nounits',
    ]
    gpu_rc, gpu_text = capture(gpu_query)

gpus = []
if gpu_rc == 0 and gpu_text:
    for row in csv.reader(gpu_text.splitlines(), skipinitialspace=True):
        if len(row) >= 5:
            gpus.append({
                'index': int(row[0]),
                'name': row[1].strip(),
                'total_mib': float(row[2]),
                'free_mib': float(row[3]),
                'driver': row[4].strip(),
                'compute_capability': row[5].strip() if len(row) > 5 else 'unknown',
            })

smi_rc, smi_full = capture(['nvidia-smi'])
nvcc_rc, nvcc_text = capture(['nvcc', '--version'])
nvcc_match = re.search(r'release\s+(\d+)\.(\d+)', nvcc_text, re.IGNORECASE)
nvcc_release = f'{nvcc_match.group(1)}.{nvcc_match.group(2)}' if nvcc_match else None
driver_cuda_match = re.search(r'CUDA Version:\s*([0-9.]+)', smi_full) if smi_rc == 0 else None
meminfo = read_meminfo()
disk_base = LIGHTNING_STUDIO_ROOT if IS_LIGHTNING else Path('/content') if IS_COLAB else Path.cwd()
disk = shutil.disk_usage(disk_base)
ENV_REPORT = {
    'platform': PLATFORM_NAME,
    'os': platform.platform(),
    'system': platform.system(),
    'machine': platform.machine(),
    'notebook_python': platform.python_version(),
    'nvidia_available': bool(gpus),
    'gpus': gpus,
    'nvidia_smi': smi_full if smi_rc == 0 else None,
    'nvcc_available': nvcc_rc == 0,
    'nvcc_version': nvcc_text if nvcc_rc == 0 else None,
    'nvcc_release': nvcc_release,
    'driver_reported_cuda_version': driver_cuda_match.group(1) if driver_cuda_match else None,
    'system_ram_total_gib': bytes_to_gib(meminfo.get('MemTotal', 0)),
    'system_ram_available_gib': bytes_to_gib(meminfo.get('MemAvailable', 0)),
    'disk_path': str(disk_base),
    'disk_total_gib': bytes_to_gib(disk.total),
    'disk_free_gib': bytes_to_gib(disk.free),
}
print(json.dumps(ENV_REPORT, indent=2))
if not gpus:
    action = 'switch the Studio machine to an A100 40GB or L40S 48GB' if IS_LIGHTNING else 'choose Runtime > Change runtime type > GPU' if IS_COLAB else 'attach a compatible NVIDIA GPU'
    print(f'BLOCKED: no NVIDIA GPU is visible. Please {action}, then rerun this cell.')

## 2. Persistent workspace setup

Lightning AI stores the complete benchmark under `/teamspace/studios/this_studio/furniture_image_to_3d_benchmark`, which persists when the Studio sleeps or changes GPU. Colab users may optionally mount Drive for inputs and outputs.

In [ ]:
MOUNT_DRIVE = False
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/furniture_image_to_3d_benchmark'

if MOUNT_DRIVE and not IS_COLAB:
    raise RuntimeError('MOUNT_DRIVE is a Colab-only option. Leave it False on Lightning AI.')

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except ImportError as exc:
        raise RuntimeError('Google Drive mounting is available only in Colab.') from exc

if IS_LIGHTNING:
    WORK_ROOT = LIGHTNING_STUDIO_ROOT / 'furniture_image_to_3d_benchmark'
elif IS_COLAB:
    WORK_ROOT = Path('/content/furniture_image_to_3d_benchmark')
else:
    WORK_ROOT = Path.cwd() / 'furniture_image_to_3d_benchmark_runtime'
PERSIST_ROOT = Path(DRIVE_PROJECT_DIR) if MOUNT_DRIVE else WORK_ROOT
INPUT_ROOT = PERSIST_ROOT / 'inputs'
OUTPUT_ROOT = PERSIST_ROOT / 'outputs'
REPO_ROOT = WORK_ROOT / 'model_repos'
ENV_ROOT = WORK_ROOT / '.venvs'
HF_HOME = WORK_ROOT / 'cache' / 'huggingface'
LOG_ROOT = WORK_ROOT / 'logs'
SCRIPT_ROOT = WORK_ROOT / 'scripts'
for directory in (WORK_ROOT, PERSIST_ROOT, INPUT_ROOT, OUTPUT_ROOT, REPO_ROOT, ENV_ROOT, HF_HOME, LOG_ROOT, SCRIPT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)
print(json.dumps({
    'platform': PLATFORM_NAME,
    'work_root': str(WORK_ROOT),
    'persistent_root': str(PERSIST_ROOT),
    'inputs': str(INPUT_ROOT),
    'outputs': str(OUTPUT_ROOT),
}, indent=2))

## 3. Model selection

This saved notebook is preconfigured with `MODEL_CHOICE='spar3d'` and `SPAR3D_LOW_VRAM=True`. On Lightning AI, select an **A100 40GB** or **L40S 48GB** before running. The cell refuses incompatible choices before cloning or installing anything.

In [ ]:
MODEL_CHOICE = 'spar3d'  # auto, trellis2, hunyuan3d_2_1, stable_fast_3d, spar3d
SPAR3D_LOW_VRAM = True
SAFETY_DISK_GIB = 2.0

MODEL_SPECS = {
    'trellis2': {
        'display_name': 'Microsoft TRELLIS.2-4B',
        'repo': 'https://github.com/microsoft/TRELLIS.2.git',
        'repo_revision': '75fbf0183001ed9876c8dbb35de6b68552ee08bd',
        'hf_id': 'microsoft/TRELLIS.2-4B',
        'hf_revision': 'af44b45f2e35a493886929c6d786e563ec68364d',
        'min_free_vram_gib': 24.0,
        'min_disk_gib': 40.0,
        'gated': False,
        'requires_nvcc': True,
        'min_nvcc': (12, 4),
        'licence': 'MIT; review dependency licences',
    },
    'hunyuan3d_2_1': {
        'display_name': 'Tencent Hunyuan3D 2.1 full PBR',
        'repo': 'https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1.git',
        'repo_revision': '82920d643c0dc2f7bfd7255f45f62d386edfe60c',
        'hf_id': 'tencent/Hunyuan3D-2.1',
        'hf_revision': '0b94677654c57bb9a6b6845cd7b704ccf551d327',
        'min_free_vram_gib': 29.0,
        'min_disk_gib': 45.0,
        'gated': False,
        'requires_nvcc': True,
        'min_nvcc': (12, 4),
        'licence': 'Tencent Hunyuan 3D 2.1 Community Licence; territory/MAU/disclosure review required',
    },
    'stable_fast_3d': {
        'display_name': 'Stability AI Stable Fast 3D',
        'repo': 'https://github.com/Stability-AI/stable-fast-3d.git',
        'repo_revision': 'ff21fc491b4dc5314bf6734c7c0dabd86b5f5bb2',
        'hf_id': 'stabilityai/stable-fast-3d',
        'hf_revision': 'f0c9a8ffd62cb1bbc8a7a53c9f87a0be1b6be778',
        'min_free_vram_gib': 6.0,
        'min_disk_gib': 15.0,
        'gated': True,
        'requires_nvcc': False,
        'min_nvcc': None,
        'licence': 'Stability AI Community Licence; enterprise licence above US$1M annual revenue',
    },
    'spar3d': {
        'display_name': 'Stability AI SPAR3D (optional)',
        'repo': 'https://github.com/Stability-AI/stable-point-aware-3d.git',
        'repo_revision': 'fdc311b16809e6a8adc2f5a3407ebb3db1a95bd1',
        'hf_id': 'stabilityai/stable-point-aware-3d',
        'hf_revision': '5699918cb34f55cd7d828493d2725f3038313761',
        'min_free_vram_gib': 7.0 if SPAR3D_LOW_VRAM else 10.5,
        'min_disk_gib': 20.0,
        'gated': True,
        'requires_nvcc': False,
        'min_nvcc': None,
        'licence': 'Stability AI Community Licence; enterprise licence above US$1M annual revenue',
    },
}

if ENV_REPORT['system'] != 'Linux':
    raise RuntimeError('BLOCKED: this cloud benchmark requires Linux. Do not install these CUDA pipelines on the current OS.')
if not ENV_REPORT['gpus']:
    raise RuntimeError('BLOCKED: no NVIDIA GPU is available. No model was selected and TripoSR was not used.')
gpu = ENV_REPORT['gpus'][0]
free_vram_gib = gpu['free_mib'] / 1024
disk_free_gib = ENV_REPORT['disk_free_gib']

def numeric_version(value):
    return tuple(int(part) for part in re.findall(r'\d+', value or '0'))

def compatibility(model_name):
    spec = MODEL_SPECS[model_name]
    reasons = []
    if free_vram_gib < spec['min_free_vram_gib']:
        reasons.append(f"needs {spec['min_free_vram_gib']:.1f} GiB free VRAM; only {free_vram_gib:.1f} GiB is free")
    if disk_free_gib < spec['min_disk_gib'] + SAFETY_DISK_GIB:
        reasons.append(f"needs about {spec['min_disk_gib']:.0f} GiB disk plus safety margin; only {disk_free_gib:.1f} GiB is free")
    if numeric_version(gpu['driver']) < (525, 60, 13):
        reasons.append(f"CUDA 12.x wheels require Linux NVIDIA driver >=525.60.13; found {gpu['driver']}")
    if spec['requires_nvcc'] and not ENV_REPORT['nvcc_available']:
        reasons.append('requires the CUDA toolkit/nvcc to compile official extensions')
    elif spec['min_nvcc'] and numeric_version(ENV_REPORT['nvcc_release']) < spec['min_nvcc']:
        required = '.'.join(map(str, spec['min_nvcc']))
        reasons.append(f"requires nvcc >= {required}; found {ENV_REPORT['nvcc_release'] or 'unknown'}")
    return not reasons, reasons

decision_log = []
if MODEL_CHOICE == 'auto':
    SELECTED_MODEL = None
    for candidate in ('trellis2', 'hunyuan3d_2_1', 'stable_fast_3d'):
        ok, reasons = compatibility(candidate)
        decision_log.append({'model': candidate, 'compatible': ok, 'reasons': reasons})
        if ok:
            SELECTED_MODEL = candidate
            break
    if SELECTED_MODEL is None:
        raise RuntimeError('BLOCKED: no candidate fits this runtime. TripoSR was not selected. Details: ' + json.dumps(decision_log))
else:
    if MODEL_CHOICE not in MODEL_SPECS:
        raise ValueError(f'Unknown MODEL_CHOICE: {MODEL_CHOICE}')
    SELECTED_MODEL = MODEL_CHOICE
    ok, reasons = compatibility(SELECTED_MODEL)
    decision_log.append({'model': SELECTED_MODEL, 'compatible': ok, 'reasons': reasons})
    if not ok:
        raise RuntimeError('BLOCKED before installation: ' + '; '.join(reasons))

SELECTED_SPEC = MODEL_SPECS[SELECTED_MODEL]
print(json.dumps({
    'selected_model': SELECTED_MODEL,
    'display_name': SELECTED_SPEC['display_name'],
    'gpu': gpu,
    'free_vram_gib': round(free_vram_gib, 2),
    'selection_log': decision_log,
    'reason': 'First compatible model in the mandated fallback order' if MODEL_CHOICE == 'auto' else 'Explicit compatible manual selection',
}, indent=2))

## 4. Dependency installation

Only the selected model is installed. The notebook uses isolated Python 3.10 environments inside the persistent benchmark folder; the only notebook-runtime package added is pinned `uv`, used to create those environments. Official repository revisions and the authors' dependency pins are preserved. Compiled-extension output is saved to `logs/install_<model>.log`.

A runtime restart is normally **not** required because inference runs in a subprocess using the isolated environment. Lightning persists the installed environment, repositories, weights, and outputs. Never continue past a failed compiler/CUDA check.

In [ ]:
import shlex
from collections import deque

UV_VERSION = '0.7.19'
MODEL_REPO = REPO_ROOT / SELECTED_MODEL
MODEL_ENV = ENV_ROOT / SELECTED_MODEL
TOOLS_ENV = ENV_ROOT / 'benchmark_tools'
INSTALL_LOG = LOG_ROOT / f'install_{SELECTED_MODEL}.log'

def run_logged(command, *, cwd=None, env=None, log_path=INSTALL_LOG):
    print('+', ' '.join(shlex.quote(str(part)) for part in command), flush=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    recent = deque(maxlen=80)
    with log_path.open('a', encoding='utf-8') as stream:
        stream.write('\nCOMMAND: ' + ' '.join(map(str, command)) + '\n')
        process = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            stream.write(line)
            stream.flush()
            recent.append(line.rstrip())
        returncode = process.wait()
    output_tail = '\n'.join(recent)
    if returncode != 0:
        raise RuntimeError(f'Dependency/compilation command failed with exit code {returncode}. Log: {log_path}\n{output_tail}')
    return output_tail

if not shutil.which('uv'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', f'uv=={UV_VERSION}'], check=True)
UV = shutil.which('uv') or str(Path(sys.executable).parent / 'uv')

if not MODEL_REPO.exists():
    run_logged(['git', 'clone', '--recursive', '--filter=blob:none', SELECTED_SPEC['repo'], str(MODEL_REPO)])
run_logged(['git', 'fetch', '--depth', '1', 'origin', SELECTED_SPEC['repo_revision']], cwd=MODEL_REPO)
run_logged(['git', 'checkout', '--detach', SELECTED_SPEC['repo_revision']], cwd=MODEL_REPO)
run_logged(['git', 'submodule', 'update', '--init', '--recursive'], cwd=MODEL_REPO)

if not MODEL_ENV.exists():
    run_logged([UV, 'venv', '--python', '3.10', '--seed', str(MODEL_ENV)])
MODEL_PYTHON = MODEL_ENV / 'bin' / 'python'
MODEL_PIP = [str(MODEL_PYTHON), '-m', 'pip']
run_logged(MODEL_PIP + ['install', '--upgrade', 'pip==25.1.1', 'wheel==0.45.1', 'setuptools==69.5.1'])

torch_index = 'https://download.pytorch.org/whl/cu124'
if SELECTED_MODEL == 'trellis2':
    run_logged(MODEL_PIP + ['install', 'torch==2.6.0', 'torchvision==0.21.0', '--index-url', torch_index])
    shell = f'source {shlex.quote(str(MODEL_ENV / "bin" / "activate"))} && . ./setup.sh --basic --flash-attn --nvdiffrast --nvdiffrec --cumesh --o-voxel --flexgemm'
    run_logged(['bash', '-lc', shell], cwd=MODEL_REPO)
elif SELECTED_MODEL == 'hunyuan3d_2_1':
    run_logged(MODEL_PIP + ['install', 'torch==2.5.1', 'torchvision==0.20.1', 'torchaudio==2.5.1', '--index-url', torch_index])
    run_logged(MODEL_PIP + ['install', '-r', str(MODEL_REPO / 'requirements.txt')], cwd=MODEL_REPO)
    run_logged(MODEL_PIP + ['install', '-e', str(MODEL_REPO / 'hy3dpaint' / 'custom_rasterizer')], cwd=MODEL_REPO)
    activation = shlex.quote(str(MODEL_ENV / 'bin' / 'activate'))
    run_logged(['bash', '-lc', f'source {activation} && bash compile_mesh_painter.sh'], cwd=MODEL_REPO / 'hy3dpaint' / 'DifferentiableRenderer')
    esrgan = MODEL_REPO / 'hy3dpaint' / 'ckpt' / 'RealESRGAN_x4plus.pth'
    esrgan.parent.mkdir(parents=True, exist_ok=True)
    if not esrgan.is_file():
        run_logged(['wget', '-O', str(esrgan), 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth'])
else:
    run_logged(MODEL_PIP + ['install', 'torch==2.5.1', 'torchvision==0.20.1', '--index-url', torch_index])
    requirements_command = MODEL_PIP + ['install', '-r', str(MODEL_REPO / 'requirements.txt')]
    requirements_env = None
    if SELECTED_MODEL == 'spar3d':
        # AlphaCLIP's isolated build environment can miss the already-installed torch/setuptools stack.
        requirements_command.insert(4, '--no-build-isolation')
        requirements_env = os.environ.copy()
        requirements_env.update({'USE_CUDA': '1', 'CUDA_HOME': '/usr/local/cuda', 'MAX_JOBS': '2'})
    run_logged(requirements_command, cwd=MODEL_REPO, env=requirements_env)
    if SELECTED_MODEL == 'spar3d':
        # transparent-background 1.3.3 imports a Flet API removed by newer releases.
        run_logged(MODEL_PIP + ['install', 'flet==0.24.1'])

cuda_smoke = (
    "import json, torch; "
    "assert torch.cuda.is_available(), 'torch.cuda.is_available() is false'; "
    "print(json.dumps({'torch': torch.__version__, 'torch_cuda': torch.version.cuda, "
    "'device': torch.cuda.get_device_name(0), 'capability': torch.cuda.get_device_capability(0)}))"
)
try:
    cuda_details = run_logged([str(MODEL_PYTHON), '-c', cuda_smoke])
except RuntimeError as exc:
    raise RuntimeError('CUDA compatibility check failed after dependency installation. Stop before downloading weights. ' + str(exc)) from exc
print('CUDA smoke test:', cuda_details.strip().splitlines()[-1])

if not TOOLS_ENV.exists():
    run_logged([UV, 'venv', '--python', '3.10', '--seed', str(TOOLS_ENV)], log_path=LOG_ROOT / 'install_tools.log')
TOOLS_PYTHON = TOOLS_ENV / 'bin' / 'python'
run_logged([str(TOOLS_PYTHON), '-m', 'pip', 'install',
            'numpy==1.26.4', 'Pillow==10.4.0', 'trimesh==4.4.7',
            'matplotlib==3.9.2', 'rembg==2.0.65', 'onnxruntime==1.20.1'],
           log_path=LOG_ROOT / 'install_tools.log')
print(f'Installed {SELECTED_MODEL} in {MODEL_ENV}')
print(f'Installation log: {INSTALL_LOG}')

## 5. Model download

Weights and required DINOv2 image encoders are downloaded at pinned Hugging Face revisions before offline inference. SF3D/SPAR3D are gated: first accept the model licence in the linked browser page, then enter a read-only token at the hidden prompt. The token remains only in process memory and is neither displayed nor written to the notebook.

In [ ]:
from getpass import getpass

HF_TOKEN = None
if SELECTED_SPEC['gated']:
    print(f"Accept access first: https://huggingface.co/{SELECTED_SPEC['hf_id']}")
    HF_TOKEN = getpass('Hugging Face read token (input hidden): ').strip()
    if not HF_TOKEN:
        raise RuntimeError('A read token is required for this gated model. No weights were downloaded.')

download_env = os.environ.copy()
download_env['HF_HOME'] = str(HF_HOME)
if HF_TOKEN:
    download_env['HF_TOKEN'] = HF_TOKEN
MODEL_SNAPSHOT = WORK_ROOT / 'weights' / SELECTED_MODEL
MODEL_SNAPSHOT.mkdir(parents=True, exist_ok=True)
download_script = (
    'import os, sys; '
    'from huggingface_hub import snapshot_download; '
    'snapshot_download(repo_id=sys.argv[1], revision=sys.argv[2], local_dir=sys.argv[3], '
    "token=os.environ.get('HF_TOKEN') or None)"
)
completed = subprocess.run(
    [str(MODEL_PYTHON), '-c', download_script, SELECTED_SPEC['hf_id'], SELECTED_SPEC['hf_revision'], str(MODEL_SNAPSHOT)],
    env=download_env, capture_output=True, text=True, check=False,
)
if completed.returncode != 0:
    detail = (completed.stdout + completed.stderr)[-5000:]
    if '401' in detail or '403' in detail or 'gated' in detail.lower():
        raise RuntimeError('Hugging Face authentication/access failed. Accept the licence and use a read token.\n' + detail)
    raise RuntimeError('Model weights are unavailable or the download failed.\n' + detail)
print(f"Downloaded and pinned {SELECTED_SPEC['hf_id']} at {SELECTED_SPEC['hf_revision']} to {MODEL_SNAPSHOT}")
DINO_SNAPSHOT = None
if SELECTED_MODEL == 'hunyuan3d_2_1':
    dino_id = 'facebook/dinov2-giant'
    dino_revision = '611a9d42f2335e0f921f1e313ad3c1b7178d206d'
    DINO_SNAPSHOT = WORK_ROOT / 'weights' / 'dinov2-giant'
    DINO_SNAPSHOT.mkdir(parents=True, exist_ok=True)
    dino_completed = subprocess.run(
        [str(MODEL_PYTHON), '-c', download_script, dino_id, dino_revision, str(DINO_SNAPSHOT)],
        env=download_env, capture_output=True, text=True, check=False,
    )
    if dino_completed.returncode != 0:
        detail = (dino_completed.stdout + dino_completed.stderr)[-5000:]
        raise RuntimeError('Pinned DINOv2 dependency download failed.\n' + detail)
    print(f'Downloaded and pinned {dino_id} at {dino_revision} to {DINO_SNAPSHOT}')
DINO_LARGE_SNAPSHOT = None
if SELECTED_MODEL in {'stable_fast_3d', 'spar3d'}:
    dino_large_id = 'facebook/dinov2-large'
    dino_large_revision = '47b73eefe95e8d44ec3623f8890bd894b6ea2d6c'
    DINO_LARGE_SNAPSHOT = WORK_ROOT / 'weights' / 'dinov2-large'
    DINO_LARGE_SNAPSHOT.mkdir(parents=True, exist_ok=True)
    dino_large_completed = subprocess.run(
        [str(MODEL_PYTHON), '-c', download_script, dino_large_id, dino_large_revision, str(DINO_LARGE_SNAPSHOT)],
        env=download_env, capture_output=True, text=True, check=False,
    )
    if dino_large_completed.returncode != 0:
        detail = (dino_large_completed.stdout + dino_large_completed.stderr)[-5000:]
        raise RuntimeError('Pinned DINOv2-Large dependency download failed.\n' + detail)
    patched_configs = []
    for config_path in list(MODEL_SNAPSHOT.rglob('*.yaml')) + list(MODEL_SNAPSHOT.rglob('*.yml')) + list(MODEL_SNAPSHOT.rglob('*.json')):
        config_text = config_path.read_text(encoding='utf-8')
        if dino_large_id in config_text:
            config_path.write_text(config_text.replace(dino_large_id, str(DINO_LARGE_SNAPSHOT)), encoding='utf-8')
            patched_configs.append(str(config_path.relative_to(MODEL_SNAPSHOT)))
    if not patched_configs:
        raise RuntimeError(f'{SELECTED_MODEL} snapshot does not reference the expected {dino_large_id} dependency; stop and review the pinned model revision.')
    print(f'Downloaded and pinned {dino_large_id} at {dino_large_revision} to {DINO_LARGE_SNAPSHOT}')
    print('Patched offline dependency path in:', patched_configs)
CLIP_SNAPSHOT = None
CLIP_REVISION = None
if SELECTED_MODEL == 'stable_fast_3d':
    clip_id = 'openai/clip-vit-large-patch14'
    CLIP_REVISION = '32bd64288804d66eefd0ccbe215aa642df71cc41'
    cache_download_script = (
        'import os, sys; from huggingface_hub import snapshot_download; '
        "print(snapshot_download(repo_id=sys.argv[1], revision=sys.argv[2], "
        "token=os.environ.get('HF_TOKEN') or None, "
        "allow_patterns=['*.json', '*.txt', '*.model', '*.safetensors']))"
    )
    clip_completed = subprocess.run(
        [str(MODEL_PYTHON), '-c', cache_download_script, clip_id, CLIP_REVISION],
        env=download_env, capture_output=True, text=True, check=False,
    )
    if clip_completed.returncode != 0:
        detail = (clip_completed.stdout + clip_completed.stderr)[-5000:]
        raise RuntimeError('Pinned CLIP dependency download failed.\n' + detail)
    CLIP_SNAPSHOT = Path(clip_completed.stdout.strip().splitlines()[-1])
    if not (CLIP_SNAPSHOT / 'model.safetensors').is_file():
        raise RuntimeError(f'Pinned CLIP snapshot is incomplete: {CLIP_SNAPSHOT}')
    print(f'Downloaded and cached {clip_id} at {CLIP_REVISION} in {CLIP_SNAPSHOT}')
HF_TOKEN = None
download_env.pop('HF_TOKEN', None)

## 6. Input image upload and reproducible manifest

Use the same five source images for every model. Filenames must have these stems: `chair_thin_legs`, `sofa`, `table`, `metallic_furniture`, and `textured_fabric_or_wood`. PNG, JPEG, and WebP are accepted. In Lightning AI, upload the included `benchmark_inputs` folder beside this notebook; the cell copies and validates the files automatically. SHA-256 hashes are stored in `inputs/manifest.json`.

In [ ]:
UPLOAD_NEW_IMAGES = True
INPUT_SOURCE_DIR = Path.cwd() / 'benchmark_inputs'
REQUIRED_CASES = {
    'chair_thin_legs': 'Chair with thin legs',
    'sofa': 'Sofa',
    'table': 'Table',
    'metallic_furniture': 'Furniture with metallic parts',
    'textured_fabric_or_wood': 'Detailed fabric or wood texture',
}
ALLOWED_IMAGE_SUFFIXES = {'.png', '.jpg', '.jpeg', '.webp'}

if UPLOAD_NEW_IMAGES:
    if IS_COLAB:
        from google.colab import files
        uploaded = files.upload()
        for filename, content in uploaded.items():
            safe_name = Path(filename).name
            if Path(safe_name).suffix.lower() not in ALLOWED_IMAGE_SUFFIXES:
                print(f'Skipping unsupported file: {safe_name}')
                continue
            (INPUT_ROOT / safe_name).write_bytes(content)
    else:
        INPUT_SOURCE_DIR.mkdir(parents=True, exist_ok=True)
        copied = []
        for source in INPUT_SOURCE_DIR.iterdir():
            if source.is_file() and source.suffix.lower() in ALLOWED_IMAGE_SUFFIXES:
                destination = INPUT_ROOT / source.name
                if source.resolve() != destination.resolve():
                    shutil.copy2(source, destination)
                copied.append(source.name)
        print(f'Copied {len(copied)} image(s) from {INPUT_SOURCE_DIR} to {INPUT_ROOT}')

available = [p for p in INPUT_ROOT.iterdir() if p.is_file() and p.suffix.lower() in ALLOWED_IMAGE_SUFFIXES]
by_stem = {p.stem: p for p in available}
missing = [stem for stem in REQUIRED_CASES if stem not in by_stem]
if missing:
    instruction = f'Upload the five images into {INPUT_SOURCE_DIR} using the Lightning file browser, then rerun this cell.' if IS_LIGHTNING else f'Place the five images in {INPUT_SOURCE_DIR}, then rerun this cell.'
    raise RuntimeError('Missing required benchmark image stems: ' + ', '.join(missing) + '\n' + instruction)

manifest = []
for stem, challenge in REQUIRED_CASES.items():
    path = by_stem[stem]
    manifest.append({
        'input_name': stem,
        'challenge': challenge,
        'source_path': str(path),
        'sha256': hashlib.sha256(path.read_bytes()).hexdigest(),
        'bytes': path.stat().st_size,
    })
MANIFEST_PATH = INPUT_ROOT / 'manifest.json'
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps(manifest, indent=2))

## 7. Image preprocessing and optional background removal

Background removal is separated from inference and is rerunnable. Processed inputs are normalized to transparent 512×512 PNGs with EXIF orientation applied. Disable removal to measure the model's own robustness to backgrounds.

In [ ]:
REMOVE_BACKGROUND = True
FOREGROUND_RATIO = 0.85
PROCESSED_ROOT = INPUT_ROOT / 'processed'
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

PREPROCESS_SCRIPT = r'''
import json
import sys
from pathlib import Path
from PIL import Image, ImageOps

source = Path(sys.argv[1])
destination = Path(sys.argv[2])
remove_background = sys.argv[3] == '1'
ratio = float(sys.argv[4])
image = ImageOps.exif_transpose(Image.open(source)).convert('RGBA')
if remove_background:
    try:
        from rembg import remove
        image = remove(image, alpha_matting=True, alpha_matting_foreground_threshold=240,
                       alpha_matting_background_threshold=10, alpha_matting_erode_size=0)
    except Exception as exc:
        raise RuntimeError(f'Background removal failed: {exc}') from exc
alpha = image.getchannel('A')
bbox = alpha.getbbox()
if bbox:
    image = image.crop(bbox)
target = max(1, int(512 * ratio))
scale = min(target / max(1, image.width), target / max(1, image.height))
size = (max(1, round(image.width * scale)), max(1, round(image.height * scale)))
image = image.resize(size, Image.Resampling.LANCZOS)
canvas = Image.new('RGBA', (512, 512), (0, 0, 0, 0))
canvas.alpha_composite(image, ((512 - image.width) // 2, (512 - image.height) // 2))
destination.parent.mkdir(parents=True, exist_ok=True)
canvas.save(destination, optimize=True)
print(json.dumps({'source': str(source), 'destination': str(destination), 'size': canvas.size}))
'''
preprocess_path = SCRIPT_ROOT / 'preprocess_image.py'
preprocess_path.write_text(PREPROCESS_SCRIPT, encoding='utf-8')

manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
for item in manifest:
    destination = PROCESSED_ROOT / f"{item['input_name']}.png"
    completed = subprocess.run([str(TOOLS_PYTHON), str(preprocess_path), item['source_path'], str(destination),
                                '1' if REMOVE_BACKGROUND else '0', str(FOREGROUND_RATIO)],
                               capture_output=True, text=True, check=False)
    if completed.returncode != 0:
        raise RuntimeError((completed.stdout + completed.stderr)[-4000:])
    item['processed_path'] = str(destination)
    item['processed_sha256'] = hashlib.sha256(destination.read_bytes()).hexdigest()
    item['background_removed'] = REMOVE_BACKGROUND
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(f'Prepared {len(manifest)} images in {PROCESSED_ROOT}')

## 8. 3D inference

Each input runs in a fresh subprocess so CUDA memory is released even after failure. Inference output is streamed live so long model-loading steps do not look frozen. Wall time includes model loading and export; the report also records baseline, peak used, and estimated incremental GPU memory. A CUDA OOM stops the run and reports the model/input explicitly.

TRELLIS uses the official O-Voxel pipeline, Hunyuan follows the official shape then PBR paint APIs, and SF3D/SPAR3D use their official `run.py`. These APIs generate their textured mesh representation as part of inference. Raw GLBs stay under the ephemeral work root until the next, independently rerunnable export-normalization section copies them into the benchmark output tree.

In [ ]:
import threading
import time

SF_TEXTURE_RESOLUTION = (
    512
    if SELECTED_MODEL in {'stable_fast_3d', 'spar3d'} and numeric_version(gpu['compute_capability']) < (8, 0)
    else 1024
)
HUNYUAN_TEXTURE_RESOLUTION = 512
HUNYUAN_MAX_VIEWS = 6
TRELLIS_DECIMATION_TARGET = 250000
BENCHMARK_SEED = 42

TRELLIS_RUNNER = r'''
import os, sys, time
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
from pathlib import Path
from PIL import Image
import torch
from trellis2.pipelines import Trellis2ImageTo3DPipeline
import o_voxel
input_path, output_path, model_path = sys.argv[1], Path(sys.argv[2]), sys.argv[3]
decimation, seed = int(sys.argv[4]), int(sys.argv[5])
output_path.parent.mkdir(parents=True, exist_ok=True)
pipeline = Trellis2ImageTo3DPipeline.from_pretrained(model_path)
pipeline.cuda()
torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
mesh = pipeline.run(Image.open(input_path).convert('RGBA'), seed=seed)[0]
mesh.simplify(16777216)
glb = o_voxel.postprocess.to_glb(
    vertices=mesh.vertices, faces=mesh.faces, attr_volume=mesh.attrs, coords=mesh.coords,
    attr_layout=mesh.layout, voxel_size=mesh.voxel_size,
    aabb=[[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],
    decimation_target=decimation, texture_size=4096, remesh=True,
    remesh_band=1, remesh_project=0, verbose=True)
glb.export(str(output_path), extension_webp=False)
print(f'INFERENCE_SECONDS={time.perf_counter() - started:.6f}')
print(f'TORCH_PEAK_MIB={torch.cuda.max_memory_allocated() / 1024 / 1024:.2f}')
'''

HUNYUAN_RUNNER = r'''
import gc, os, random, shutil, sys, time
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
from pathlib import Path
import numpy as np
import torch
import huggingface_hub
repo, input_path, output_path = Path(sys.argv[1]), sys.argv[2], Path(sys.argv[3])
model_id, model_snapshot, dino_snapshot = sys.argv[4], Path(sys.argv[5]), Path(sys.argv[6])
resolution, max_views, seed = int(sys.argv[7]), int(sys.argv[8]), int(sys.argv[9])
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
original_snapshot_download = huggingface_hub.snapshot_download
def pinned_snapshot_download(repo_id, *args, **kwargs):
    if repo_id == model_id:
        return str(model_snapshot)
    return original_snapshot_download(repo_id, *args, **kwargs)
huggingface_hub.snapshot_download = pinned_snapshot_download
sys.path.insert(0, str(repo / 'hy3dshape'))
sys.path.insert(0, str(repo / 'hy3dpaint'))
sys.path.insert(0, str(repo))
from PIL import Image
from hy3dshape.pipelines import Hunyuan3DDiTFlowMatchingPipeline
from textureGenPipeline import Hunyuan3DPaintPipeline, Hunyuan3DPaintConfig
output_path.parent.mkdir(parents=True, exist_ok=True)
untextured = output_path.with_name('model_untextured.glb')
textured_obj = output_path.with_name('model_textured.obj')
torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
shape_pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(str(model_snapshot))
generator = torch.Generator(device='cuda').manual_seed(seed)
mesh = shape_pipeline(image=Image.open(input_path).convert('RGBA'), generator=generator)[0]
mesh.export(untextured)
del mesh, shape_pipeline
gc.collect(); torch.cuda.empty_cache()
config = Hunyuan3DPaintConfig(max_num_view=max_views, resolution=resolution)
config.realesrgan_ckpt_path = str(repo / 'hy3dpaint/ckpt/RealESRGAN_x4plus.pth')
config.multiview_cfg_path = str(repo / 'hy3dpaint/cfgs/hunyuan-paint-pbr.yaml')
config.custom_pipeline = str(repo / 'hy3dpaint/hunyuanpaintpbr')
config.multiview_pretrained_path = model_id
config.dino_ckpt_path = str(dino_snapshot)
torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
paint_pipeline = Hunyuan3DPaintPipeline(config)
paint_pipeline(mesh_path=str(untextured), image_path=input_path, output_mesh_path=str(textured_obj), save_glb=True)
generated_glb = textured_obj.with_suffix('.glb')
if not generated_glb.is_file():
    raise RuntimeError(f'Hunyuan paint stage did not create {generated_glb}')
shutil.copy2(generated_glb, output_path)
print(f'INFERENCE_SECONDS={time.perf_counter() - started:.6f}')
print(f'TORCH_PEAK_MIB={torch.cuda.max_memory_allocated() / 1024 / 1024:.2f}')
'''

trellis_runner_path = SCRIPT_ROOT / 'run_trellis2.py'
hunyuan_runner_path = SCRIPT_ROOT / 'run_hunyuan3d_2_1.py'
SEEDED_OFFICIAL_RUNNER = r'''
import random, runpy, sys
from pathlib import Path
import numpy as np
import torch
runner, seed = sys.argv[1], int(sys.argv[2])
sys.path.insert(0, str(Path(runner).resolve().parent))
sys.argv = [runner] + sys.argv[3:]
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
runpy.run_path(runner, run_name='__main__')
'''
seeded_runner_path = SCRIPT_ROOT / 'run_official_seeded.py'
trellis_runner_path.write_text(TRELLIS_RUNNER, encoding='utf-8')
hunyuan_runner_path.write_text(HUNYUAN_RUNNER, encoding='utf-8')
seeded_runner_path.write_text(SEEDED_OFFICIAL_RUNNER, encoding='utf-8')

def current_gpu_used_mib():
    rc, text = capture(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'])
    if rc != 0 or not text:
        return None
    try:
        return float(text.splitlines()[0].strip())
    except ValueError:
        return None

def run_monitored(command, *, cwd, env):
    samples = []
    baseline = current_gpu_used_mib()
    stop = threading.Event()
    def monitor():
        while not stop.wait(0.2):
            value = current_gpu_used_mib()
            if value is not None:
                samples.append(value)
    watcher = threading.Thread(target=monitor, daemon=True)
    started = time.perf_counter()
    watcher.start()
    output_lines = []
    process = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            output_lines.append(line)
        returncode = process.wait()
    finally:
        stop.set(); watcher.join(timeout=2)
    wall = time.perf_counter() - started
    output = ''.join(output_lines)
    peak = max(samples) if samples else current_gpu_used_mib()
    torch_match = re.findall(r'TORCH_PEAK_MIB=([0-9.]+)|Peak Memory:\s*([0-9.]+)', output)
    torch_values = [float(a or b) for a, b in torch_match]
    inference_match = re.search(r'INFERENCE_SECONDS=([0-9.]+)', output)
    return {
        'returncode': returncode,
        'output': output,
        'wall_time_seconds_including_model_load': wall,
        'reported_inference_seconds': float(inference_match.group(1)) if inference_match else None,
        'baseline_gpu_used_mib': baseline,
        'peak_gpu_used_mib': peak,
        'estimated_incremental_peak_mib': max(0.0, peak - baseline) if peak is not None and baseline is not None else None,
        'torch_peak_allocated_mib': max(torch_values) if torch_values else None,
    }

runtime_env = os.environ.copy()
runtime_env.update({
    'HF_HOME': str(HF_HOME),
    'HF_HUB_OFFLINE': '1',
    'TRANSFORMERS_OFFLINE': '1',
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
    'PYTHONHASHSEED': str(BENCHMARK_SEED),
})
official_runner = MODEL_REPO / 'run.py'
t4_fp16_math_sdpa_compat = False
if SELECTED_MODEL in {'stable_fast_3d', 'spar3d'} and numeric_version(gpu['compute_capability']) < (8, 0):
    official_source = official_runner.read_text(encoding='utf-8')
    if 'dtype=torch.bfloat16' not in official_source:
        raise RuntimeError(f'Pinned {SELECTED_MODEL} runner no longer contains the expected BF16 autocast; review before applying the T4 compatibility patch.')
    patched_source = official_source.replace('dtype=torch.bfloat16', 'dtype=torch.float16')
    reduction_patch = (
        "import torch\n"
        "if hasattr(torch.backends.cuda, 'allow_fp16_bf16_reduction_math_sdp'):\n"
        "    torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(True)\n"
    )
    patched_source = patched_source.replace('import torch\n', reduction_patch, 1)
    official_runner = MODEL_REPO / 'run_t4_fp16.py'
    compile(patched_source, str(official_runner), 'exec')
    official_runner.write_text(patched_source, encoding='utf-8')
    t4_fp16_math_sdpa_compat = True
    print(f'T4 compatibility: {SELECTED_MODEL} will use FP16 autocast and reduced-precision math SDPA.')

if SELECTED_MODEL in {'stable_fast_3d', 'spar3d'}:
    rc, free_text = capture(['nvidia-smi', '--query-gpu=memory.free', '--format=csv,noheader,nounits'])
    if rc != 0 or not free_text:
        raise RuntimeError('Could not read current free GPU memory before inference.')
    free_now_gib = float(free_text.splitlines()[0].strip()) / 1024
    if t4_fp16_math_sdpa_compat:
        required_free_gib = 12.0
    elif SELECTED_MODEL == 'spar3d':
        required_free_gib = 7.0 if SPAR3D_LOW_VRAM else 10.5
    else:
        required_free_gib = 6.0
    print(f'Pre-inference free VRAM: {free_now_gib:.2f} GiB; required: {required_free_gib:.1f} GiB')
    if free_now_gib < required_free_gib:
        raise RuntimeError(
            f'Only {free_now_gib:.2f} GiB of GPU memory is free, but {SELECTED_MODEL} requires about '
            f'{required_free_gib:.1f} GiB in this mode. Restart the Studio/runtime to clear stale GPU processes.'
        )

manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
for item in manifest:
    input_name = item['input_name']
    output_dir = OUTPUT_ROOT / SELECTED_MODEL / input_name
    output_dir.mkdir(parents=True, exist_ok=True)
    raw_dir = WORK_ROOT / 'raw_inference' / SELECTED_MODEL / input_name
    if raw_dir.exists():
        shutil.rmtree(raw_dir)
    raw_dir.mkdir(parents=True, exist_ok=True)
    raw_glb_path = raw_dir / 'mesh.glb'
    if SELECTED_MODEL == 'trellis2':
        command = [str(MODEL_PYTHON), str(trellis_runner_path), item['processed_path'], str(raw_glb_path),
                   str(MODEL_SNAPSHOT), str(TRELLIS_DECIMATION_TARGET), str(BENCHMARK_SEED)]
    elif SELECTED_MODEL == 'hunyuan3d_2_1':
        command = [str(MODEL_PYTHON), str(hunyuan_runner_path), str(MODEL_REPO), item['processed_path'],
                   str(raw_glb_path), SELECTED_SPEC['hf_id'], str(MODEL_SNAPSHOT), str(DINO_SNAPSHOT),
                   str(HUNYUAN_TEXTURE_RESOLUTION), str(HUNYUAN_MAX_VIEWS), str(BENCHMARK_SEED)]
    else:
        command = [str(MODEL_PYTHON), str(seeded_runner_path), str(official_runner),
                   str(BENCHMARK_SEED), item['processed_path'],
                   '--output-dir', str(raw_dir), '--texture-resolution', str(SF_TEXTURE_RESOLUTION),
                   '--pretrained-model', str(MODEL_SNAPSHOT)]
        if SELECTED_MODEL == 'spar3d' and SPAR3D_LOW_VRAM:
            command.append('--low-vram-mode')
    print(f'Running {SELECTED_MODEL} on {input_name}...')
    metrics = run_monitored(command, cwd=MODEL_REPO, env=runtime_env)
    log_path = output_dir / 'inference.log'
    log_path.write_text(metrics.pop('output'), encoding='utf-8')
    if metrics['returncode'] != 0:
        detail = log_path.read_text(encoding='utf-8')[-6000:]
        if 'out of memory' in detail.lower() or 'cuda oom' in detail.lower():
            raise RuntimeError(f'CUDA out of memory for {SELECTED_MODEL}/{input_name}. No fallback was attempted.\n{detail}')
        raise RuntimeError(f'Inference failed for {SELECTED_MODEL}/{input_name}. See {log_path}.\n{detail}')
    generated = raw_dir / '0' / 'mesh.glb' if SELECTED_MODEL in {'stable_fast_3d', 'spar3d'} else raw_glb_path
    if not generated.is_file() or generated.stat().st_size <= 0:
        raise RuntimeError(f'Official runner completed but did not create a non-empty textured GLB at {generated}')
    metrics.update({
        'model_name': SELECTED_MODEL,
        'model_display_name': SELECTED_SPEC['display_name'],
        'input_name': input_name,
        'input_sha256': item['sha256'],
        'processed_sha256': item['processed_sha256'],
        'gpu_name': gpu['name'],
        'gpu_total_mib': gpu['total_mib'],
        'repo_revision': SELECTED_SPEC['repo_revision'],
        'weights_revision': SELECTED_SPEC['hf_revision'],
        'auxiliary_weights_revision': {
            'hunyuan3d_2_1': 'facebook/dinov2-giant@611a9d42f2335e0f921f1e313ad3c1b7178d206d',
            'stable_fast_3d': [
                'facebook/dinov2-large@47b73eefe95e8d44ec3623f8890bd894b6ea2d6c',
                'openai/clip-vit-large-patch14@32bd64288804d66eefd0ccbe215aa642df71cc41',
            ],
            'spar3d': 'facebook/dinov2-large@47b73eefe95e8d44ec3623f8890bd894b6ea2d6c',
        }.get(SELECTED_MODEL),
        'random_seed': BENCHMARK_SEED,
        'texture_resolution': SF_TEXTURE_RESOLUTION if SELECTED_MODEL in {'stable_fast_3d', 'spar3d'} else HUNYUAN_TEXTURE_RESOLUTION if SELECTED_MODEL == 'hunyuan3d_2_1' else None,
        't4_fp16_math_sdpa_compat': t4_fp16_math_sdpa_compat if SELECTED_MODEL in {'stable_fast_3d', 'spar3d'} else False,
        'raw_textured_glb_path': str(generated),
        'raw_textured_glb_bytes': generated.stat().st_size,
    })
    (output_dir / 'runtime_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    print(json.dumps(metrics, indent=2))

## 9. Textured GLB export normalization

This rerunnable step copies the raw official GLB into `outputs/<model>/<input>/model.glb`, computes its SHA-256, and updates runtime metrics. It does not reconstruct the object again. TRELLIS explicitly disables WebP texture export for broader GLB-to-USDZ portability.

In [ ]:
for item in json.loads(MANIFEST_PATH.read_text(encoding='utf-8')):
    input_name = item['input_name']
    raw_dir = WORK_ROOT / 'raw_inference' / SELECTED_MODEL / input_name
    generated = raw_dir / '0' / 'mesh.glb' if SELECTED_MODEL in {'stable_fast_3d', 'spar3d'} else raw_dir / 'mesh.glb'
    if not generated.is_file() or generated.stat().st_size <= 0:
        raise RuntimeError(f'Raw textured GLB is missing or empty: {generated}. Rerun inference for this input.')
    output_dir = OUTPUT_ROOT / SELECTED_MODEL / input_name
    output_dir.mkdir(parents=True, exist_ok=True)
    glb_path = output_dir / 'model.glb'
    shutil.copy2(generated, glb_path)
    metrics_path = output_dir / 'runtime_metrics.json'
    if not metrics_path.is_file():
        raise RuntimeError(f'Runtime metrics are missing: {metrics_path}')
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    metrics['glb_bytes'] = glb_path.stat().st_size
    metrics['glb_sha256'] = hashlib.sha256(glb_path.read_bytes()).hexdigest()
    metrics_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    print(input_name, metrics['glb_bytes'], 'bytes', metrics['glb_sha256'])

## 10. Mesh validation, PBR inspection, and preview image

GLB is the authoritative AR asset. Validation does not require watertightness because open surfaces may be legitimate, but the value is recorded. Invalid/empty meshes, non-finite vertices, broken indices, missing materials/base-colour textures, external GLB resources, or failed GLB loading fail the report. Degenerate faces are counted and surfaced as warnings because a small number can occur in otherwise loadable exports.

In [ ]:
VALIDATION_SCRIPT = r'''
import json, math, struct, sys, traceback
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import trimesh

GLB_JSON = 0x4E4F534A
def read_glb_json(path):
    with path.open('rb') as stream:
        header = stream.read(12)
        if len(header) != 12:
            raise ValueError('Incomplete GLB header')
        magic, version, declared = struct.unpack('<4sII', header)
        if magic != b'glTF' or version != 2 or declared != path.stat().st_size:
            raise ValueError('Invalid GLB 2.0 header or declared length')
        while stream.tell() < declared:
            length, chunk_type = struct.unpack('<II', stream.read(8))
            chunk = stream.read(length)
            if len(chunk) != length:
                raise ValueError('Incomplete GLB chunk')
            if chunk_type == GLB_JSON:
                return json.loads(chunk.rstrip(b'\x00 \t\r\n').decode('utf-8'))
    raise ValueError('GLB JSON chunk missing')

output_dir = Path(sys.argv[1])
glb_path = output_dir / 'model.glb'
report = {'path': str(glb_path), 'exists': glb_path.is_file(), 'file_size_bytes': glb_path.stat().st_size if glb_path.is_file() else 0, 'valid': False}
try:
    if not glb_path.is_file() or glb_path.stat().st_size <= 0:
        raise ValueError('GLB file is missing or empty')
    document = read_glb_json(glb_path)
    images = document.get('images', [])
    buffers = document.get('buffers', [])
    materials = document.get('materials', [])
    embedded = [image for image in images if 'bufferView' in image or str(image.get('uri', '')).startswith('data:')]
    external = [image.get('uri') for image in images if image.get('uri') and not str(image['uri']).startswith('data:')]
    external_buffers = [buffer.get('uri') for buffer in buffers if buffer.get('uri') and not str(buffer['uri']).startswith('data:')]
    material_reports = []
    for index, material in enumerate(materials):
        pbr = material.get('pbrMetallicRoughness', {})
        material_reports.append({
            'index': index,
            'name': material.get('name') or f'material_{index}',
            'base_color_factor': pbr.get('baseColorFactor', [1, 1, 1, 1]),
            'base_color_texture': pbr.get('baseColorTexture'),
            'roughness_factor': pbr.get('roughnessFactor', 1.0),
            'metallic_factor': pbr.get('metallicFactor', 1.0),
            'metallic_roughness_texture': pbr.get('metallicRoughnessTexture'),
            'normal_texture': material.get('normalTexture'),
            'occlusion_texture': material.get('occlusionTexture'),
            'emissive_texture': material.get('emissiveTexture'),
            'alpha_mode': material.get('alphaMode', 'OPAQUE'),
        })
    scene = trimesh.load(glb_path, force='scene', process=False)
    geometries = [geometry for geometry in scene.geometry.values() if isinstance(geometry, trimesh.Trimesh)]
    if not geometries:
        raise ValueError('GLB loaded but contains no triangle mesh')
    vertex_count = sum(len(mesh.vertices) for mesh in geometries)
    face_count = sum(len(mesh.faces) for mesh in geometries)
    connected_components = 0
    watertight = []
    winding = []
    degenerate = 0
    invalid_indices = 0
    nonfinite_vertices = 0
    for mesh in geometries:
        vertices = np.asarray(mesh.vertices)
        faces = np.asarray(mesh.faces)
        nonfinite_vertices += int((~np.isfinite(vertices)).any(axis=1).sum())
        if len(faces):
            invalid_indices += int(((faces < 0) | (faces >= len(vertices))).any(axis=1).sum())
            triangles = vertices[faces]
            cross = np.cross(triangles[:, 1] - triangles[:, 0], triangles[:, 2] - triangles[:, 0])
            scale = max(float(np.ptp(vertices, axis=0).max()), 1.0)
            degenerate += int((np.linalg.norm(cross, axis=1) <= scale * scale * 1e-12).sum())
        connected_components += len(mesh.split(only_watertight=False))
        watertight.append(bool(mesh.is_watertight))
        winding.append(bool(mesh.is_winding_consistent))
    base_color_textures = sum(item['base_color_texture'] is not None for item in material_reports)
    pbr_support = bool(material_reports and base_color_textures and all(
        item['roughness_factor'] is not None and item['metallic_factor'] is not None for item in material_reports))
    validation_errors = []
    validation_warnings = []
    if vertex_count <= 0 or face_count <= 0:
        validation_errors.append('Mesh has no usable vertices or faces')
    if invalid_indices:
        validation_errors.append(f'{invalid_indices} faces contain invalid vertex indices')
    if nonfinite_vertices:
        validation_errors.append(f'{nonfinite_vertices} vertices contain non-finite values')
    if degenerate:
        validation_warnings.append(f'{degenerate} degenerate faces detected')
    if not material_reports:
        validation_errors.append('No glTF materials found')
    if not images or base_color_textures <= 0:
        validation_errors.append('No embedded base-colour texture found')
    if external or external_buffers:
        validation_errors.append('GLB references external texture or buffer resources')
    report.update({
        'glb_loading_valid': True,
        'geometry_count': len(geometries),
        'vertex_count': vertex_count,
        'face_count_polygon_count': face_count,
        'connected_components': connected_components,
        'watertight_all_components': all(watertight),
        'watertight_by_geometry': watertight,
        'winding_consistent_all': all(winding),
        'degenerate_face_count': degenerate,
        'invalid_face_index_count': invalid_indices,
        'nonfinite_vertex_count': nonfinite_vertices,
        'material_count': len(material_reports),
        'materials': material_reports,
        'image_count': len(images),
        'embedded_texture_count': len(embedded),
        'external_texture_uris': external,
        'external_buffer_uris': external_buffers,
        'image_mime_types': [image.get('mimeType') for image in images],
        'pbr_support_detected': pbr_support,
        'validation_errors': validation_errors,
        'validation_warnings': validation_warnings,
    })
    report['valid'] = not validation_errors
    report['android_ar_glb_candidate'] = bool(report['valid'] and len(embedded) == len(images) and len(images) > 0 and base_color_textures > 0 and not external_buffers)
    report['iphone_usdz_conversion_candidate'] = bool(report['android_ar_glb_candidate'] and pbr_support)
    report['ar_note'] = 'Candidate flags are technical heuristics; Android Scene Viewer and Apple Quick Look device tests remain required.'

    points = np.vstack([np.asarray(mesh.vertices) for mesh in geometries])
    if len(points) > 50000:
        rng = np.random.default_rng(1234)
        points = points[rng.choice(len(points), 50000, replace=False)]
    figure, axes = plt.subplots(1, 3, figsize=(12, 4), facecolor='#111827')
    for axis, pair, title in zip(axes, ((0, 2), (0, 1), (2, 1)), ('Front', 'Top', 'Side')):
        axis.scatter(points[:, pair[0]], points[:, pair[1]], s=0.15, c='#67e8f9', alpha=0.65)
        axis.set_aspect('equal', adjustable='box'); axis.axis('off'); axis.set_title(title, color='white')
    figure.tight_layout()
    figure.savefig(output_dir / 'preview.png', dpi=180, bbox_inches='tight', facecolor=figure.get_facecolor())
    plt.close(figure)
except Exception as exc:
    report['error'] = str(exc)
    report['traceback'] = traceback.format_exc()[-4000:]
(output_dir / 'validation_report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
print(json.dumps(report, indent=2))
sys.exit(0 if report.get('valid') else 1)
'''
validation_path = SCRIPT_ROOT / 'validate_model.py'
validation_path.write_text(VALIDATION_SCRIPT, encoding='utf-8')

for item in json.loads(MANIFEST_PATH.read_text(encoding='utf-8')):
    output_dir = OUTPUT_ROOT / SELECTED_MODEL / item['input_name']
    completed = subprocess.run([str(TOOLS_PYTHON), str(validation_path), str(output_dir)], capture_output=True, text=True, check=False)
    if completed.returncode != 0:
        print(completed.stdout[-4000:] + completed.stderr[-4000:])
        raise RuntimeError(f"Validation failed for {SELECTED_MODEL}/{item['input_name']}. See {output_dir / 'validation_report.json'}.")
    report = json.loads((output_dir / 'validation_report.json').read_text(encoding='utf-8'))
    print(item['input_name'], {key: report.get(key) for key in (
        'valid', 'vertex_count', 'face_count_polygon_count', 'connected_components',
        'watertight_all_components', 'degenerate_face_count', 'embedded_texture_count',
        'material_count', 'pbr_support_detected', 'android_ar_glb_candidate',
        'iphone_usdz_conversion_candidate')})

## 11. OBJ export where supported

OBJ/MTL cannot represent glTF metallic/roughness PBR faithfully, so GLB remains authoritative. This best-effort, independently rerunnable step exports only a single-geometry GLB; multi-geometry assets are reported as unsupported instead of flattening or losing their material structure.

In [ ]:
OBJ_EXPORT_SCRIPT = r'''
import json, sys
from pathlib import Path
import trimesh
from trimesh.exchange.obj import export_obj

output_dir = Path(sys.argv[1])
glb_path = output_dir / 'model.glb'
report = {'supported': False}
try:
    scene = trimesh.load(glb_path, force='scene', process=False)
    geometries = [geometry for geometry in scene.geometry.values() if isinstance(geometry, trimesh.Trimesh)]
    if len(geometries) != 1:
        report['reason'] = f'Skipped: GLB contains {len(geometries)} geometries; preserving GLB material structure takes priority.'
    else:
        obj_text, resources = export_obj(geometries[0], include_normals=True, include_color=True,
                                         include_texture=True, return_texture=True, mtl_name='model.mtl')
        (output_dir / 'model.obj').write_text(obj_text, encoding='utf-8')
        texture_dir = output_dir / 'textures'
        texture_dir.mkdir(exist_ok=True)
        texture_names = [name for name in resources if Path(name).suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'}]
        written = []
        for name, data in resources.items():
            payload = data.encode('utf-8') if isinstance(data, str) else data
            if Path(name).suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'}:
                destination = texture_dir / Path(name).name
                destination.write_bytes(payload)
            else:
                text = payload.decode('utf-8', errors='replace')
                for texture_name in texture_names:
                    text = text.replace(texture_name, 'textures/' + Path(texture_name).name)
                destination = output_dir / Path(name).name
                destination.write_text(text, encoding='utf-8')
            written.append(str(destination.relative_to(output_dir)))
        report = {'supported': True, 'obj_bytes': (output_dir / 'model.obj').stat().st_size,
                  'resource_files': sorted(written)}
except Exception as exc:
    report = {'supported': False, 'error': str(exc)}
(output_dir / 'obj_export_report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
validation_path = output_dir / 'validation_report.json'
if validation_path.is_file():
    validation = json.loads(validation_path.read_text(encoding='utf-8'))
    validation['obj_export'] = report
    validation_path.write_text(json.dumps(validation, indent=2), encoding='utf-8')
print(json.dumps(report, indent=2))
'''
obj_export_path = SCRIPT_ROOT / 'export_obj.py'
obj_export_path.write_text(OBJ_EXPORT_SCRIPT, encoding='utf-8')
for item in json.loads(MANIFEST_PATH.read_text(encoding='utf-8')):
    output_dir = OUTPUT_ROOT / SELECTED_MODEL / item['input_name']
    completed = subprocess.run([str(TOOLS_PYTHON), str(obj_export_path), str(output_dir)], capture_output=True, text=True, check=False)
    if completed.returncode != 0:
        raise RuntimeError((completed.stdout + completed.stderr)[-4000:])
    print(item['input_name'], completed.stdout.strip())

## 12. Interactive 3D preview

Choose an output below. The cell embeds GLB bytes into a browser-side `<model-viewer>` and refuses very large files to avoid crashing the notebook tab. The saved `preview.png` remains a lightweight three-view geometry diagnostic.

In [ ]:
import base64
from IPython.display import HTML, display

PREVIEW_INPUT_NAME = 'chair_thin_legs'
preview_glb = OUTPUT_ROOT / SELECTED_MODEL / PREVIEW_INPUT_NAME / 'model.glb'
if not preview_glb.is_file():
    raise FileNotFoundError(preview_glb)
if preview_glb.stat().st_size > 40 * 1024 * 1024:
    raise RuntimeError('Interactive embedding is limited to 40 MB. Download the GLB and inspect it in a native viewer.')
payload = base64.b64encode(preview_glb.read_bytes()).decode('ascii')
display(HTML(f'''
<script type='module' src='https://unpkg.com/@google/model-viewer/dist/model-viewer.min.js'></script>
<model-viewer src='data:model/gltf-binary;base64,{payload}' camera-controls auto-rotate shadow-intensity='1'
  style='width:100%;height:520px;background:#111827;border-radius:12px'></model-viewer>
'''))

## 13. Comparison table and human review

Automated reports cannot see whether hidden geometry is plausible or whether the source colour/texture was preserved perceptually. Edit `human_scores.json` after inspecting each GLB and source image. Scores are 1–5. Re-run this cell after every model/runtime; Drive persistence lets it aggregate all available models.

In [ ]:
from IPython.display import Markdown

score_path = PERSIST_ROOT / 'human_scores.json'
all_reports = list(OUTPUT_ROOT.glob('*/*/validation_report.json'))
scores = json.loads(score_path.read_text(encoding='utf-8')) if score_path.exists() else {}
added_score_rows = False
for report_path in all_reports:
    key = f'{report_path.parent.parent.name}/{report_path.parent.name}'
    if key not in scores:
        scores[key] = {
            'geometry_completeness_1_to_5': None,
            'thin_part_preservation_1_to_5': None,
            'texture_accuracy_1_to_5': None,
            'colour_preservation_1_to_5': None,
            'notes': '',
        }
        added_score_rows = True
score_path.write_text(json.dumps(scores, indent=2), encoding='utf-8')
if added_score_rows:
    print(f'Added unscored rows to {score_path}. Fill them, then rerun this cell.')

licence_labels = {
    'trellis2': 'Potentially suitable: MIT; audit dependency licences',
    'hunyuan3d_2_1': 'Conditional: territory, MAU, disclosure and AUP legal review',
    'stable_fast_3d': 'Conditional: Community Licence revenue threshold / enterprise licence',
    'spar3d': 'Conditional: Community Licence revenue threshold / enterprise licence',
}
def average(values):
    valid = [float(value) for value in values if isinstance(value, (int, float))]
    return round(sum(valid) / len(valid), 2) if valid else 'Not scored'

rows = []
for model_dir in sorted(path for path in OUTPUT_ROOT.iterdir() if path.is_dir()):
    validations = []
    runtimes = []
    model_scores = []
    for input_dir in sorted(path for path in model_dir.iterdir() if path.is_dir()):
        validation_file = input_dir / 'validation_report.json'
        runtime_file = input_dir / 'runtime_metrics.json'
        if validation_file.is_file() and runtime_file.is_file():
            validations.append(json.loads(validation_file.read_text(encoding='utf-8')))
            runtimes.append(json.loads(runtime_file.read_text(encoding='utf-8')))
            model_scores.append(scores.get(f'{model_dir.name}/{input_dir.name}', {}))
    if not validations:
        continue
    passed = sum(bool(item.get('valid')) for item in validations)
    pbr = sum(bool(item.get('pbr_support_detected')) for item in validations)
    android = sum(bool(item.get('android_ar_glb_candidate')) for item in validations)
    iphone = sum(bool(item.get('iphone_usdz_conversion_candidate')) for item in validations)
    peak_values = [item.get('torch_peak_allocated_mib') or item.get('estimated_incremental_peak_mib') for item in runtimes]
    peak_values = [value for value in peak_values if isinstance(value, (int, float))]
    geometry_score = average([item.get('geometry_completeness_1_to_5') for item in model_scores])
    thin_score = average([item.get('thin_part_preservation_1_to_5') for item in model_scores])
    texture_score = average([item.get('texture_accuracy_1_to_5') for item in model_scores])
    colour_score = average([item.get('colour_preservation_1_to_5') for item in model_scores])
    quality_scores = [geometry_score, thin_score, texture_score, colour_score]
    score_fields = ('geometry_completeness_1_to_5', 'thin_part_preservation_1_to_5',
                    'texture_accuracy_1_to_5', 'colour_preservation_1_to_5')
    human_complete = bool(model_scores) and all(
        isinstance(score.get(field), (int, float)) and 1 <= score[field] <= 5
        for score in model_scores for field in score_fields
    )
    if not human_complete:
        recommendation = 'Pending human scores and real Android/iPhone device validation'
    elif passed < len(validations):
        recommendation = 'Do not advance yet: one or more GLBs failed technical validation'
    elif pbr < len(validations):
        recommendation = 'Do not advance yet: PBR support was incomplete'
    elif sum(quality_scores) / len(quality_scores) >= 4.0:
        recommendation = 'Technical finalist: proceed to Android/iPhone device tests and legal review'
    elif sum(quality_scores) / len(quality_scores) >= 3.0:
        recommendation = 'Conditional: inspect failure notes before device and legal review'
    else:
        recommendation = 'Do not advance: human quality scores are below the benchmark threshold'
    row = {
        'model_name': model_dir.name,
        'gpu_used': ', '.join(sorted({item.get('gpu_name', 'unknown') for item in runtimes})),
        'peak_vram_mib': round(max(peak_values), 1) if peak_values else 'Unavailable',
        'generation_time_seconds_total': round(sum(item.get('wall_time_seconds_including_model_load', 0) for item in runtimes), 2),
        'geometry_completeness': geometry_score,
        'thin_part_preservation': thin_score,
        'texture_accuracy': texture_score,
        'colour_preservation': colour_score,
        'pbr_support': f'{pbr}/{len(validations)} passed',
        'glb_quality': f'{passed}/{len(validations)} valid',
        'suitability_for_android_ar': f'{android}/{len(validations)} technical candidates; device test required',
        'suitability_for_iphone_usdz_conversion': f'{iphone}/{len(validations)} conversion candidates; Quick Look test required',
        'licence_suitability': licence_labels.get(model_dir.name, 'Legal review required'),
        'overall_recommendation': recommendation,
    }
    rows.append(row)

comparison_csv = PERSIST_ROOT / 'benchmark_comparison.csv'
comparison_md = PERSIST_ROOT / 'benchmark_comparison.md'
if rows:
    with comparison_csv.open('w', newline='', encoding='utf-8') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader(); writer.writerows(rows)
    headers = list(rows[0])
    markdown = '| ' + ' | '.join(headers) + ' |\n|' + '|'.join(['---'] * len(headers)) + '|\n'
    for row in rows:
        markdown += '| ' + ' | '.join(str(row[key]).replace('|', '/') for key in headers) + ' |\n'
    comparison_md.write_text(markdown, encoding='utf-8')
    display(Markdown(markdown))
    print(f'Saved {comparison_csv} and {comparison_md}')
else:
    print('No complete validation/runtime pairs are available yet.')

## 14. Output download

The archive contains `outputs/<model>/<input>/model.glb`, optional OBJ/MTL/textures, `preview.png`, `validation_report.json`, `runtime_metrics.json`, and inference logs.

In [ ]:
DOWNLOAD_MODEL = SELECTED_MODEL
download_source = OUTPUT_ROOT / DOWNLOAD_MODEL
if not download_source.is_dir():
    raise FileNotFoundError(download_source)
archive_base = PERSIST_ROOT / f'{DOWNLOAD_MODEL}_furniture_benchmark_outputs'
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=OUTPUT_ROOT, base_dir=DOWNLOAD_MODEL)
print(f'Created {archive_path} ({Path(archive_path).stat().st_size / 1024 / 1024:.1f} MB)')
try:
    from google.colab import files
    files.download(archive_path)
except ImportError:
    if IS_LIGHTNING:
        print('Lightning AI: download the ZIP from the Studio file browser using the path printed above.')
    else:
        print('Use the archive path printed above.')

## 15. Runtime cleanup

Inference subprocesses exit after each image, so GPU memory should already be released. The flags below are intentionally false. Never delete persisted outputs until they have been reviewed or downloaded.

In [ ]:
DELETE_SELECTED_ENV = False
DELETE_SELECTED_REPOSITORY = False
DELETE_MODEL_CACHE = False
DELETE_SELECTED_OUTPUTS = False

targets = []
if DELETE_SELECTED_ENV:
    targets.append(MODEL_ENV)
if DELETE_SELECTED_REPOSITORY:
    targets.append(MODEL_REPO)
if DELETE_MODEL_CACHE:
    targets.extend([HF_HOME, MODEL_SNAPSHOT])
    if SELECTED_MODEL == 'hunyuan3d_2_1' and DINO_SNAPSHOT is not None:
        targets.append(DINO_SNAPSHOT)
    if SELECTED_MODEL in {'stable_fast_3d', 'spar3d'} and DINO_LARGE_SNAPSHOT is not None:
        targets.append(DINO_LARGE_SNAPSHOT)
if DELETE_SELECTED_OUTPUTS:
    targets.append(OUTPUT_ROOT / SELECTED_MODEL)
for target in targets:
    target = Path(target).resolve()
    allowed_roots = [WORK_ROOT.resolve(), PERSIST_ROOT.resolve()]
    if not any(target == root or root in target.parents for root in allowed_roots):
        raise RuntimeError(f'Refusing to delete path outside benchmark roots: {target}')
    if target.exists():
        shutil.rmtree(target)
        print(f'Deleted {target}')
if not targets:
    print('No cleanup flags were enabled. GPU inference processes have already exited.')
capture(['nvidia-smi'])[1]